The preliminaries

In [6]:
from openai import OpenAI
from scipy.stats import spearmanr
import jsonschema, json
import os
from dotenv import load_dotenv

length_id_batch = 25

#Hidden OpenAI API Key 
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("API Key not found! Ensure you have a .env file with OPENAI_API_KEY defined.")

client = OpenAI(api_key=api_key)

#Pull in AIME and GPQA Diamond
with open('aime_pool.json', 'r') as f:
    aime_data = json.load(f)

# Load the GPQA pool
with open('gpqa_pool.json', 'r') as f:
    gpqa_data = json.load(f)

# Load the GPQA pool
with open('asudit.json', 'r') as f:
    asudit_data = json.load(f)

# Verify the data loaded correctly
print(f"Loaded {len(aime_data)} items from AIME")
print(f"Loaded {len(gpqa_data)} items from GPQA")
print(f"Loaded {len(asudit_data)} items from asudit")

#split case_ids
gpqa_id = asudit_data["case_ids"][:length_id_batch]
aime_id = asudit_data["case_ids"][length_id_batch:]

gpqa_data_filter = {}
gpqa_responses = {}

for case in gpqa_data:
    if case['case_id'] in gpqa_id:
        gpqa_data_filter[case['case_id']] = case
        gpqa_responses[case['case_id']] = {}

aime_data_filter = {}
aime_responses = {}
for case in aime_data:
    if case['case_id'] in aime_id:
        aime_data_filter[case['case_id']] = case
        aime_responses[case['case_id']] = {}

#print(gpqa_id)
#print(aime_id)
print(len(gpqa_data_filter), len(aime_data_filter))

Loaded 196 items from AIME
Loaded 198 items from GPQA
Loaded 4 items from asudit
25 25


1. Builds a prompt and queries the model K = 50 times with temperature=1.0 passed explicitly on every call (do not omit the parameter and rely on the API default). The exact model(s) you call are determined by your axis assignment (see the table in Your Assignment below).

In [ ]:
#Axis C mid-vs-frontier: gpt-4.1-mini (zero-shot) [Condition A] vs gpt-4.1 (zero-shot) [Condition B]
import copy
from concurrent.futures import ThreadPoolExecutor
from pydantic import BaseModel, Field
from typing import Literal

#models = ["gpt-4.1-mini", "gpt-4.1"]
#benchmarks = ['gpqa_diamond', 'aime']
#ground_truth = ['correct_letter', 'correct_answer']

# Schema for GPQA (Multiple Choice)
class GPQAResponse(BaseModel):
    answer: Literal["A", "B", "C", "D"]

# Schema for AIME (Integer)
class AIMEResponse(BaseModel):
    answer: int = Field(ge=0, le=999) # Ensures the integer is between 0 and 999

def generate_gpqa_prompt(subject):
    """
    Subject: Physics, Chemistry, or Biology
    """
    # Using an f-string to inject the 'subject' variable
    system_prompt_gpqa = f"""You are a PhD student at a top tier research university, studying {subject}.
                        Goal: Using your expert knowledge in {subject}, answer the following multiple choice questions.
                        Rules:
                        - Select one letter answer.
                        - Single-agent only. No tool-use, no multi-turn, no external retrieval.
                        Output format:
                        1. A single letter corresponding to your answer"""
    return system_prompt_gpqa

system_prompt_aime = """You are a mathematics professor at a top tier research university.
Goal: Using your expert knowledge in mathematics, solve the following math problems from the American Invitational Mathematics Examination.
Rules:
- Return a single integer answer in the range [0, 999].
- Single-agent only. No tool-use, no multi-turn, no external retrieval.
Output format:
1. A single integer in the range [0, 999] corresponding to your answer"""


def run_agent(system_prompt, model, user_message, case_id, repository, response_schema, iterations=50):
    messages = [{"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message},]
    repository[case_id]['answers'] = []
    
    def get_completion(_):
        try:
            resp = client.beta.chat.completions.parse(
                model=model, 
                messages=messages, 
                max_tokens=500, 
                temperature=1.0,
                response_format=response_schema # The Pydantic class
            )
            return resp.choices[0].message.parsed.answer
        except Exception as e:
            print(f"Error in Case {case_id}: {e}")
            return "_UNPARSEABLE_"
    # Use ThreadPoolExecutor to run iterations in parallel
    # max_workers=20 is a safe starting point to avoid hitting rate limits too fast
    with ThreadPoolExecutor(max_workers=20) as executor:
        results = list(executor.map(get_completion, range(iterations)))

    repository[case_id]['answers'].extend(results)
    print(f"Completed {iterations} iterations for Case {case_id} using {model}")
    
   
        
       
gpqa_repo = {"gpt-4.1-mini":copy.deepcopy(gpqa_responses), "gpt-4.1":copy.deepcopy(gpqa_responses)}
for model in gpqa_repo.keys():
    for id in gpqa_id:
        system_prompt_gpqa = generate_gpqa_prompt(gpqa_data_filter[id]['subject'])
        run_agent(system_prompt=system_prompt_gpqa,
                  model=model,
                  user_message=gpqa_data_filter[id]["question"],
                  case_id=id,
                  repository=gpqa_repo[model],
                  response_schema=GPQAResponse,
                  iterations=50)

    # Save after all 25 GPQA cases for this model are done
    with open(f"raw_gpqa_{model}_results.json", "w") as f:
        json.dump(gpqa_repo[model], f, indent=4)

aime_repo = {"gpt-4.1-mini":copy.deepcopy(aime_responses), "gpt-4.1":copy.deepcopy(aime_responses)}
for model in aime_repo.keys():
    for id in aime_id:
        #system_prompt_gpqa = generate_gpqa_prompt(gpqa_data_filter[id]['subject'])
        run_agent(system_prompt=system_prompt_aime,
                  model=model,
                  user_message=aime_data_filter[id]["question"],
                  case_id=id,
                  repository=aime_repo[model],
                  response_schema=AIMEResponse,
                  iterations=50)
    
    # Save after all 25 AIME cases for this model are done
    with open(f"raw_aime_{model}_results.json", "w") as f:
        json.dump(aime_repo[model], f, indent=4)





Completed 50 iterations for Case gpqa_d_157 using gpt-4.1-mini
Completed 50 iterations for Case gpqa_d_151 using gpt-4.1-mini
Completed 50 iterations for Case gpqa_d_032 using gpt-4.1-mini
Completed 50 iterations for Case gpqa_d_096 using gpt-4.1-mini
Completed 50 iterations for Case gpqa_d_101 using gpt-4.1-mini
Completed 50 iterations for Case gpqa_d_015 using gpt-4.1-mini
Completed 50 iterations for Case gpqa_d_072 using gpt-4.1-mini
Completed 50 iterations for Case gpqa_d_002 using gpt-4.1-mini
Completed 50 iterations for Case gpqa_d_113 using gpt-4.1-mini
Completed 50 iterations for Case gpqa_d_075 using gpt-4.1-mini
Completed 50 iterations for Case gpqa_d_156 using gpt-4.1-mini
Completed 50 iterations for Case gpqa_d_016 using gpt-4.1-mini
Completed 50 iterations for Case gpqa_d_165 using gpt-4.1-mini
Completed 50 iterations for Case gpqa_d_190 using gpt-4.1-mini
Completed 50 iterations for Case gpqa_d_028 using gpt-4.1-mini
Completed 50 iterations for Case gpqa_d_120 using gpt-4

2. Parses each response to extract a clean answer (a single letter "A"-"D" for GPQA, a Python int in [0, 999] for AIME). Responses you cannot parse should collapse to the literal string "_UNPARSEABLE_" — the catch-all bucket for malformed model output. _UNPARSEABLE_ entries count as wrong when computing accuracy A.

3. Computes per-case metrics — consistency C, accuracy A, risk-adjusted accuracy 𝑅𝛼, and the Wilson lower bound R_wilson (formulas in The Data → Metrics below).

4. Validates and serializes the results into a single {PennKey}_results.json (for example, dkaihua_results.json) that satisfies the provided JSON schema, then computes per-cell Spearman for the report.